# 03. What the measurements can and cannot seeRun `make all` first. This notebook reads `results/`.Three questions in order: which parameters move which measurements (sensitivity), whichcombinations are invisible (identifiability), and whether a maneuver fixes it (tie-breaker).

In [ ]:
from pathlib import Pathimport matplotlib.pyplot as pltimport numpy as npimport pandas as pdfrom hcmtwin.analysis.identifiability import HIDDEN_ORDERfrom hcmtwin.viz import stylestyle.apply()pd.set_option("display.width", 200)RESULTS = Path("../results")

## 1. SensitivityThe first result is visible without any inference at all: the parameter that mostdetermines the outcome does not move any baseline measurement.

In [ ]:
sobol = pd.read_csv(RESULTS / "sensitivity_sobol.csv")outcome = (sobol[sobol["quantity"] == "ef_drop_at_mid_dose"]           .sort_values("ST", ascending=False))display(outcome[["parameter", "group", "S1", "ST"]].head(8).round(3))display(pd.read_csv(RESULTS / "sensitivity_summary.csv").round(3))

## 2. Identifiability

In [ ]:
fisher = pd.read_csv(RESULTS / "fisher_table.csv")realistic = fisher[fisher["noise_level"] == "realistic"]eigen = [c for c in realistic.columns if c.startswith("eigenvalue_")]print("median Fisher eigenvalue spectrum (largest first):")print(realistic[eigen].median().round(6).to_string())print()weights = [c for c in realistic.columns if c.startswith("invisible_weight_")]composition = realistic[weights].median()composition.index = [c.replace("invisible_weight_", "") for c in composition.index]print("composition of the least-visible direction:")print(composition.round(4).to_string())

In [ ]:
confounding = pd.read_csv(RESULTS / "confounding_table.csv")display(confounding[confounding["noise_level"] == "realistic"]        [["parameter_a", "parameter_b", "median_abs_correlation",          "fraction_above_threshold", "confounded"]].round(3))display(pd.read_csv(RESULTS / "recovery_summary.csv").round(3))

### How well is each parameter recovered, patient by patient?A relative credible-interval width near 2 means the interval spans the prior box: the datasaid nothing at all about that parameter.

In [ ]:
recovery = pd.read_csv(RESULTS / "recovery_table.csv")realistic = recovery[recovery["noise_level"] == "realistic"]order = (realistic.groupby("parameter")["relative_ci90_width"].median()         .sort_values().index.tolist())figure, ax = plt.subplots(figsize=(7.2, 3.8))data = [realistic[realistic["parameter"] == p]["relative_ci90_width"].to_numpy()        for p in order]parts = ax.boxplot(data, vert=False, patch_artist=True, widths=0.55,                   medianprops={"color": style.TEXT_PRIMARY, "linewidth": 1.6},                   flierprops={"markersize": 3, "markerfacecolor": style.TEXT_MUTED,                               "markeredgewidth": 0})for patch in parts["boxes"]:    patch.set_facecolor(style.SERIES[0]); patch.set_alpha(0.55); patch.set_linewidth(0)ax.axvline(0.60, color=style.TEXT_MUTED, linestyle=(0, (4, 3)), linewidth=1.0)ax.set_yticklabels([p.replace("_", " ") for p in order])ax.set_xlabel("Relative width of the 90% credible interval")ax.set_title("Parameter recovery at realistic measurement error")ax.grid(axis="y", visible=False)plt.tight_layout(); plt.show()

## 3. Tie-breaker

In [ ]:
table = pd.read_csv(RESULTS / "tiebreaker_table.csv")display(table)detail = pd.read_csv(RESULTS / "tiebreaker_detail.csv")display(detail[detail["usable"]].groupby(["pair", "maneuver"])        [["correlation_before", "correlation_after", "best_signal_to_noise"]]        .median().round(3))

### The parameters no maneuver can help withThe distinction between *weakly identified* (a better measurement could help) and*structurally unidentified* (there is no signal to amplify) is the most useful thing thisanalysis produces, because the two call for completely different responses.

In [ ]:
display(pd.read_csv(RESULTS / "structural_identifiability.csv").round(4))